In [1]:
# --- Dataerai provenance (optional) -----------------------------------------
# Traces this notebook run - cell source, outputs, logs, transfers, and
# environment - to the Dataerai platform when the SDK and daemon are
# available. Without them the notebook runs exactly as before.
try:
    %load_ext dataerai.magics
    %dataerai --trace --notebook large_stem_simulations.ipynb abTEM / notebook runs / articles
except Exception as _dataerai_error:
    print(f"Dataerai tracing not active: {_dataerai_error}")


Signed in as demo@dataerai.com
Dataerai destination: abTEM / notebook runs / articles
Tracing notebook run de54b0b3-5daf-4a0e-954b-1e4a16794826. Cell source, outputs, logs, transfers, and environment details will be uploaded when %dataerai --finish runs.


# Large STEM simulations with PRISM (Au decahedron)

This notebook demonstrates efficient large-field scanning TEM with the
PRISM algorithm (`SMatrix`) on an Au decahedron nanoparticle, and
records full provenance to Dataerai.

> Modernized from the original abTEM ~1.0 article notebook to the
> abTEM 1.1 API. The original swept grid sizes up to 4096² on GPU to
> benchmark abTEM against Prismatic; here we run a single CPU-tractable
> field (no GPU required). Reference Prismatic/GPU timings from the
> original are retained below as documented constants for context.

In [2]:
import time

import matplotlib
import numpy as np
from ase.cluster import Decahedron

import abtem
from abtem import dataerai

abtem.config.set({'diagnostics.progress_bar': False})

In [3]:
# Au decahedron nanoparticle in a large field of view.
extent = 40.0  # Angstrom
atoms = Decahedron('Au', 4, 2, 0, latticeconstant=None)
atoms.rotate(30, 'x', center='cop')
atoms.cell[0, 0] = extent
atoms.cell[1, 1] = extent
atoms.center()
atoms.center(vacuum=2, axis=2)
print(f'{len(atoms)} atoms in a {extent:.0f} x {extent:.0f} A field')
atoms

85 atoms in a 40 x 40 A field


Atoms(symbols='Au85', pbc=False, cell=[40.0, 40.0, 13.993918150555368])

In [4]:
experiment = dataerai.start_run(
    name='Au decahedron PRISM STEM',
    collection='abTEM / notebook runs / articles',
)
experiment.capture_structure(atoms)

'structure-au85'

In [5]:
# Frozen-phonon potential (thermal averaging over configurations).
frozen_phonons = abtem.FrozenPhonons(
    atoms, num_configs=3, sigmas={'Au': 0.12}, seed=1
)
potential = abtem.Potential(
    frozen_phonons,
    gpts=768,
    slice_thickness=2,
    projection='infinite',
    parametrization='kirkland',
    device='cpu',
)
experiment.capture_potential(potential)
print('grid:', potential.gpts, '| sampling (A):', np.round(potential.sampling, 3))

grid: (768, 768) | sampling (A): [0.052 0.052]


In [6]:
# PRISM scattering matrix: interpolation is the PRISM speed-up factor.
s_matrix = abtem.SMatrix(
    potential=potential,
    energy=80e3,
    semiangle_cutoff=25,
    interpolation=(4, 4),
    device='cpu',
    store_on_host=True,
)
experiment.capture_illumination(s_matrix)
print('plane waves in S-matrix:', len(s_matrix))

plane waves in S-matrix: 137


In [7]:
# Nyquist-sampled scan over the full field; BF / ADF / HAADF detectors.
sampling = abtem.transfer.nyquist_sampling(
    s_matrix.semiangle_cutoff, s_matrix.energy
)
scan = abtem.GridScan(
    start=(0, 0), end=(1, 1), fractional=True,
    potential=potential, sampling=sampling,
)
bright_field = abtem.AnnularDetector(inner=0, outer=20)
adf = abtem.AnnularDetector(inner=40, outer=100)
haadf = abtem.AnnularDetector(inner=80, outer=150)
experiment.capture_scan(scan)
for det in (bright_field, adf, haadf):
    experiment.capture_detector(det)
print(f'{len(scan)} probe positions, ratio to plane waves: {len(scan) / len(s_matrix):.1f}')

9216 probe positions, ratio to plane waves: 67.3


In [8]:
# Run the PRISM STEM simulation (CPU) and time it.
start = time.time()
bf_image, adf_image, haadf_image = s_matrix.scan(
    scan=scan, detectors=[bright_field, adf, haadf]
).compute()
elapsed = time.time() - start
print(f'PRISM STEM on CPU: {elapsed:.1f} s for {len(scan)} probes')

experiment.capture_measurement(bf_image, name='bright_field')
experiment.capture_measurement(adf_image, name='adf')
experiment.capture_measurement(haadf_image, name='haadf')

PRISM STEM on CPU: 19.3 s for 9216 probes


'measurement-haadf'

In [9]:
fig, axes = matplotlib.pyplot.subplots(1, 3, figsize=(13, 4))
for ax, image, title in zip(
    axes, (bf_image, adf_image, haadf_image),
    ('Bright field', 'ADF', 'HAADF'),
):
    image.show(ax=ax, title=title)
matplotlib.pyplot.tight_layout()

In [10]:
# Reference timings from the original article (documented constants):
# Prismatic 1.2.1 binary, and abTEM GPU, at gpts 512/1024/2048.
reference_timings_s = {
    'prismatic_gpu': {512: 1.9, 1024: 5.667, 2048: 23.9},
    'prismatic_cpu': {512: 9.3, 1024: 61, 2048: 419},
}
print('This CPU run (gpts=768):', round(elapsed, 1), 's')
print('Reference:', reference_timings_s)

This CPU run (gpts=768): 19.3 s
Reference: {'prismatic_gpu': {512: 1.9, 1024: 5.667, 2048: 23.9}, 'prismatic_cpu': {512: 9.3, 1024: 61, 2048: 419}}


In [11]:
dataerai.finish_run()

In [12]:
# --- Dataerai provenance: publish the execution trace, if one is active -----
try:
    %dataerai --finish
except Exception as _dataerai_error:
    print(f"Dataerai trace not published: {_dataerai_error}")


Notebook trace will publish after this cell finishes.


Published notebook execution trace de54b0b3-5daf-4a0e-954b-1e4a16794826 (11 cells, 11 products).
